<a href="https://colab.research.google.com/github/nurfnick/Operations_Research/blob/main/ClassWork/CatanLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Programming Catan

Resources:
1. Wheat
2. etc

Game Play
1. Towns
2. City
3. Development
4. Roads

Cost:

Victory Points:
1. town = 1
2. city = 2
3. largest_army = 2
4. longest_road = 2

Objective:

Maximize victory points
$$
1*town + 2*city +2*largest_army + 2*longest_road + 1*development_vp
$$

Constraints:

    Piece Constraints:

$$ town\leq 5$$
$$ city \leq 4$$
$$roads \leq 10?$$

    Building Constraints:
I don't think this is correct.  Not sure how to do this in the LP scheme...  Looks like ``Big M" constraint.  Should look at thsi for other schemese.

$$ 2*wheat + 3* ore \geq town$$

How do you spend the resources so they aren't available for the other building to use...

    Conversion Constraints:
  
$$  sheep\geq 4*ore $$



Methinks this would work too.  If you have a port, you can adjust accordingly

$$ sheep - 4*sheepToOre - 4*sheepToWheat +OreToSheep + WheatToSheep + 3*city \geq 0 $$

In [33]:
from scipy.optimize import linprog
import numpy as np

wood = 8
sheep = 8
brick = 8
wheat = 20
ore = 20
M = 10000

#x = [settlement, city, development, longest_road, largest_army, roads]
A = np.array([[1,0,0,0,0,1],#wood
              [1,0,1,0,0,0],#sheep
              [1,0,0,0,0,1],#brick
              [1,2,1,0,0,0],#wheat
              [0,3,1,0,0,0],#ore
              [0,0,0,M,0,-1],#activate longest rd
              [0,0,-0.56,0,M,0],# activate largest army
              [0,1,0,0,0,0],#max num cities
              [0,0,1,0,0,0],
              [0,0,0,0,0,1],
              [0,0,0,1,0,0],
              [0,0,0,0,1,0],
              [-1,1,0,0,0,0],
              [1,-1,0,0,0,0]
              ]
             )
b = np.array([wood+4,sheep+2,brick+4,wheat,ore,M-5,M-3,4,25,15,1,1,0,5])#wood,sheep, brick ore, road-longest, development-army,numcity,num develop, num roads, one longest, one largest, city comes from conversion of settlement
c = np.array([-1,
              -1,#not 2 due to the settle being needed
              -1/5,
              -2,
              -2,
              0])#victory points based on x

linprog(c,A,b)

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -17.199112
              x: [ 9.000e+00  4.000e+00  1.000e+00  9.998e-01  9.998e-01
                   3.000e+00]
            nit: 4
          lower:  residual: [ 9.000e+00  4.000e+00  1.000e+00  9.998e-01
                              9.998e-01  3.000e+00]
                 marginals: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00
                              0.000e+00  0.000e+00]
          upper:  residual: [       inf        inf        inf        inf
                                    inf        inf]
                 marginals: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00
                              0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 0.000e+00  0.000e+00 ...  5.000e+00
                              0.000e+00]
                 marginals: [-2.000e-04 -2.001e-01 ... -0.0

In [34]:
from scipy.optimize import LinearConstraint

b_l = np.full_like(b, -np.inf, dtype=float)

constraints = LinearConstraint(A, b_l, b)

integrality = np.ones_like(c)

from scipy.optimize import milp

res = milp(c=c, constraints=constraints, integrality=integrality)

res

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -13.6
              x: [ 7.000e+00  4.000e+00  3.000e+00  1.000e+00  0.000e+00
                   5.000e+00]
 mip_node_count: 1
 mip_dual_bound: -13.6
        mip_gap: 0.0

Slack for majority of equations is left over resources.